In [1]:
import numpy as np 
import pandas as pd
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import accuracy_score

# Load and View Data

In [2]:
iris = pd.read_csv("iris.csv")
spambase = pd.read_csv("spambase.csv")

#processing data
mapping = {"Iris-setosa":0, "Iris-versicolor": 1, "Iris-virginica": 2}
iris_features = iris.iloc[:, :-1]
iris_labels = iris.iloc[:, -1].map(mapping)

spambase_features = spambase.iloc[:, :-1]
spambase_labels = spambase.iloc[:, -1]

# Node Class

In [3]:
class Node: 
    def __init__(self, target, feature_idx=None, threshold=None, left=None, right=None):
        self.target = target
        self.left = left
        self.right = right
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.is_leaf = False
        #for leaf node
        self.set_values()

    def set_values(self):
        unique, counts = np.unique(self.target, return_counts=True)
        self.values = {}
        for i in range(len(unique)):  
            self.values[unique[i]] = counts[i]
        if len(unique) == 1 or self.feature_idx is None:
            self.is_leaf = True

# Tree Class

In [4]:
class DecisionTree:
    def __init__(self, min_samples):
        self.root= None
        self.min_samples = min_samples

    def entropy(self, y):
        y = np.array(y)
        unique, counts = np.unique(y, return_counts=True)
        sum = 0
        for i in range(len(unique)):
            p_i = counts[i]/len(y)
            sum += -1 * p_i * np.log2(p_i)
        return sum

    def information_gain(self, feature, threshold, y):
        feature = np.array(feature)
        y_array = np.array(y)
        
        left_indices = np.where(feature <= threshold)[0]
        right_indices = np.where(feature > threshold)[0]
        
        if len(left_indices) == 0 or len(right_indices) == 0:
            return 0
            
        left = y_array[left_indices]
        right = y_array[right_indices]
        
        gain = self.entropy(y_array) - ( self.entropy(left) * len(left)/len(y_array) + self.entropy(right) * len(right)/len(y_array))
        return gain

    def best_split(self, X, y):
        best_info_gain = -float('inf')
        best_feature_idx, best_threshold = None, None

        for i in range(X.shape[1]):
            thresholds = np.unique(X.iloc[:, i])

            for threshold in thresholds:
                info_gain = self.information_gain(X.iloc[:, i], threshold, y)
                if info_gain > best_info_gain:
                    best_info_gain = info_gain
                    best_feature_idx = i
                    best_threshold = threshold

        return best_feature_idx, best_threshold
    
    def make_tree(self, X, y):
        y_array = np.array(y)
        
        #checking min_samples condition or the targets
        if len(y) <= self.min_samples or len(np.unique(y_array)) == 1:
            return Node(y_array)

        feature_idx, threshold = self.best_split(X, y)
        # no good split found
        if feature_idx is None:
            return Node(y_array)

        #node with the best possible split
        node = Node(y_array, feature_idx, threshold)
        
        left = X.iloc[:, feature_idx] <= threshold
        right = X.iloc[:, feature_idx] > threshold
        
        if sum(left) > 0:
            node.left = self.make_tree(X[left], y.iloc[left.values])
        else:
            node.left = Node(y_array)
            
        if sum(right) > 0:
            node.right = self.make_tree(X[right], y.iloc[right.values])
        else:
            node.right = Node(y_array)

        return node
            
    def fit(self, X, y):
        self.root = self.make_tree(X, y)

    def predict_sample(self,x):
        node = self.root
        while not node.is_leaf:
            if node.left is None or node.right is None:
                break
            if x.iloc[node.feature_idx] <= node.threshold:
                node = node.left
            else:
                node = node.right
        return max(node.values, key=node.values.get)
        
    def predict(self, X):
        predictions = []
        for idx, row in X.iterrows():
            predictions.append(self.predict_sample(row))
        return np.array(predictions)


# Cross-Validation

In [5]:
def cross_validate(X,y, min_samples_values):
    results = {}
    for min_samples in min_samples_values:
        kf = KFold(n_splits=10, shuffle=True, random_state=42)
        accuracies = []
        for train_index, test_index in kf.split(X):
            X_train,X_test = X.iloc[train_index, :], X.iloc[test_index, :]
            y_train,y_test = y.iloc[train_index], y.iloc[test_index]
            
            tree = DecisionTree(min_samples)
            tree.fit(X_train, y_train)
            predictions = tree.predict(X_test)
            accuracies.append(accuracy_score(y_test, predictions))
        results[min_samples] = accuracies
    return results


# Results

In [6]:
iris_results = cross_validate(iris_features, iris_labels, [5, 10, 15, 20])

print("Iris Dataset Results:")
for key, acc in iris_results.items():
    print(f"n_min: {key}, Accuracy: {np.mean(acc)}, Std Deviation: {np.std(acc)}")



Iris Dataset Results:
n_min: 5, Accuracy: 0.9395238095238095, Std Deviation: 0.04675647335346747
n_min: 10, Accuracy: 0.9466666666666667, Std Deviation: 0.04988876515698587
n_min: 15, Accuracy: 0.9466666666666667, Std Deviation: 0.04988876515698587
n_min: 20, Accuracy: 0.9466666666666667, Std Deviation: 0.04988876515698587


In [7]:
spambase_results = cross_validate(spambase_features, spambase_labels, [5, 10, 15, 20, 25])

print("Spambase Dataset Results:")
for key, acc in spambase_results.items():
    print(f"n_min: {key}, Accuracy: {np.mean(acc)}, Standard Deiation: {np.std(acc)}")

Spambase Dataset Results:
n_min: 5, Accuracy: 0.9202173913043478, Standard Deiation: 0.014290068199604182
n_min: 10, Accuracy: 0.9180434782608696, Standard Deiation: 0.015125498945042552
n_min: 15, Accuracy: 0.9202173913043478, Standard Deiation: 0.016671234408728565
n_min: 20, Accuracy: 0.9204347826086957, Standard Deiation: 0.016760291536866426
n_min: 25, Accuracy: 0.9202173913043478, Standard Deiation: 0.015976412197107073
